# Exact-time GraphSAGE GPU retraining

Retrains the existing two-layer Host/Process GraphSAGE with the same node/edge schema, using a verified cache whose cutoff equals each target event timestamp at microsecond precision. Every evidence timestamp must be strictly earlier than the target. It does not read raw logs or graph.sqlite in Kaggle. The attached dataset must be mounted at `/kaggle/input/datasets/ngkimngn/provenance-exact-time-graphsage`. Enable a GPU accelerator.


In [ ]:
from pathlib import Path
import json, sys

CACHE_ROOT = Path('/kaggle/input/datasets/ngkimngn/provenance-exact-time-graphsage')
required = ['manifest.json', 'selection.json', 'ego_cache.py', 'train_gpu.py', 'train']
missing = [name for name in required if not (CACHE_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f"Expected the attached exact-time dataset at {CACHE_ROOT}; missing: {missing}")

sys.path.insert(0, str(CACHE_ROOT))
from ego_cache import load_manifest, require_exact_time
CACHE_ROOT, meta = load_manifest(CACHE_ROOT)
require_exact_time(meta)
if meta['config']['ego'].get('max_nodes') != 64 or meta['relations'] != ['creates', 'communicates', 'communicates_reverse', 'hosts']:
    raise ValueError('Node/edge schema differs from the frozen Host/Process GraphSAGE schema')
print('Exact-time dataset:', CACHE_ROOT)
print('Time policy:', meta['time_policy'])
print('Time audit:', meta['time_audit'])
print('Graph sample:', meta['splits'])
print('Nodes/edges:', len(meta['attributes']), meta['relations'])

import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training.')


In [ ]:
import subprocess

OUTPUT = Path('/kaggle/working/exact-time-graphsage-model')
EPOCHS = 30
BATCH_SIZE = 128
RESUME = False
trainer_script = CACHE_ROOT / 'train_gpu.py'
command = [sys.executable, str(trainer_script), '--cache', str(CACHE_ROOT),
           '--output', str(OUTPUT), '--epochs', str(EPOCHS),
           '--batch-size', str(BATCH_SIZE), '--device', 'cuda', '--evaluate-test']
if RESUME:
    command.append('--resume')
print('Training with exact-time cache; validation selects threshold; TEST is evaluation only.')
subprocess.run(command, check=True)


In [ ]:
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))
assert metrics['time_policy']['quantum_us'] == 1
print('These are cached uniform-sample metrics, not a full-corpus cascade evaluation.')


In [ ]:
# Verify NumPy weight export (compatible with relational_sage.RelationalSAGE.load())
import numpy as np

numpy_path = OUTPUT / 'model_numpy.npz'
if numpy_path.is_file():
    d = np.load(numpy_path, allow_pickle=False)
    dims = d['dimensions'].tolist()
    param_keys = [k for k in d.files if not k.startswith('adam_') and k not in ('dimensions', 'steps')]
    n_params = sum(d[k].size for k in param_keys)
    print(f'model_numpy.npz: dimensions={dims}  params={n_params}  steps={int(d["steps"])}')
    print(f'Keys: {sorted(param_keys)}')
    print(f'File size: {numpy_path.stat().st_size:,} bytes')
    print()
    print('To use locally:')
    print('  from relational_sage import RelationalSAGE')
    print('  model = RelationalSAGE.load("model_numpy.npz")')
    print('  # Then run: python3 code/audit/graph_vs_ml_complementarity.py --model-weights <dir>')
else:
    print('WARNING: model_numpy.npz not found — check train_gpu.py export_numpy_weights() was called')


In [ ]:
# Package model outputs for download and local complementarity analysis.
# Bundles: checkpoint.pt, model_numpy.npz, metrics.json, config.json,
#          progress.json, and the cache manifest digest for provenance.
import hashlib, os, zipfile
from pathlib import Path

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

zip_path = Path('/kaggle/working/exact_time_graphsage_results.zip')
tmp_path = zip_path.with_suffix('.zip.tmp')

# Files to include from the model output directory
INCLUDE_SUFFIXES = {'.pt', '.npz', '.json'}

try:
    with zipfile.ZipFile(tmp_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        # Model artifacts
        for src in sorted(OUTPUT.rglob('*')):
            if src.is_file() and src.suffix in INCLUDE_SUFFIXES:
                archive.write(src, 'exact-time-graphsage-model/' + src.relative_to(OUTPUT).as_posix())
        # Cache manifest digest (provenance, not the full cache)
        manifest_path = CACHE_ROOT / 'manifest.json'
        if manifest_path.is_file():
            archive.write(manifest_path, 'provenance/exact_time_cache_manifest.json')
        # Selection.json (source_line join key for complementarity analysis)
        selection_path = CACHE_ROOT / 'selection.json'
        if selection_path.is_file():
            archive.write(selection_path, 'provenance/exact_time_selection.json')
        archive.writestr('provenance/README.txt',
            'exact-time-graphsage-model/model_numpy.npz  — NumPy weights (relational_sage.RelationalSAGE.load())\n'
            'exact-time-graphsage-model/checkpoint.pt    — PyTorch checkpoint (requires PyTorch to load)\n'
            'exact-time-graphsage-model/metrics.json     — val/test metrics and threshold\n'
            'exact-time-graphsage-model/config.json      — model hyperparameters and cache identity\n'
            'provenance/exact_time_selection.json             — test/val ego-graph source_line join keys\n'
            'Join source_line in selection.json with ML test_errors.json for complementarity analysis.\n'
            'Run: python3 code/audit/graph_vs_ml_complementarity.py --model-weights <dir with model_numpy.npz>\n'
        )
    os.replace(tmp_path, zip_path)
finally:
    if tmp_path.exists():
        tmp_path.unlink()

digest = sha256(zip_path)
size_mb = zip_path.stat().st_size / 1024**2
print(f'Created: {zip_path}')
print(f'Size:    {size_mb:.2f} MiB')
print(f'SHA256:  {digest}')
print()
print('Contents:')
with zipfile.ZipFile(zip_path) as z:
    for info in z.infolist():
        print(f'  {info.filename}  ({info.file_size:,} bytes)')
